# Этап 2c — Two-Tower retrieval (локально)

Эмбеддинги обучены в Colab (`02b_two_tower_train_colab.ipynb`), скачаны в
`data/processed/two_tower_out/`. Здесь: upsert в отдельную Qdrant-коллекцию
`two_tower_items` → top-100 retrieval с cold-start fallback → сравнение
Recall@100/NDCG@100 с ALS (`02a`) и popularity baseline.

In [1]:
import sys
sys.path.append("..")

import json
import numpy as np
import pandas as pd

from src.vector_store import VectorStore
from src.cold_start import retrieve_candidates
from src.metrics import evaluate_recommendations, evaluate_fixed_recommendation

PROCESSED_DIR = "../data/processed"
TWO_TOWER_DIR = f"{PROCESSED_DIR}/two_tower_out-2"
K = 100
EMBEDDING_DIM = 64

## Загрузка артефактов

In [2]:
train = pd.read_parquet(f"{PROCESSED_DIR}/train.parquet")
test = pd.read_parquet(f"{PROCESSED_DIR}/test.parquet")
popularity_ranking = pd.read_parquet(f"{PROCESSED_DIR}/popularity_ranking.parquet")

train_positive = train[train["is_positive"] == 1]
test_positive = test[test["is_positive"] == 1]

user_embeddings = np.load(f"{TWO_TOWER_DIR}/user_embeddings.npy")
item_embeddings = np.load(f"{TWO_TOWER_DIR}/item_embeddings.npy")

with open(f"{TWO_TOWER_DIR}/idx2user.json") as f:
    idx2user = {int(k): v for k, v in json.load(f).items()}
with open(f"{TWO_TOWER_DIR}/idx2item.json") as f:
    idx2item = {int(k): v for k, v in json.load(f).items()}

user2idx = {raw_id: idx for idx, raw_id in idx2user.items()}
known_users = set(idx2user.values())

print(f"user_embeddings {user_embeddings.shape}, item_embeddings {item_embeddings.shape}")

user_embeddings (5400, 64), item_embeddings (3662, 64)


## Загрузка item-эмбеддингов в Qdrant

Отдельная коллекция `two_tower_items` — не пересекается с `als_items` из
`02a`, оба метода можно держать в Qdrant одновременно и сравнивать.

In [3]:
store = VectorStore()
store.create_collection("two_tower_items", dim=EMBEDDING_DIM)

item_ids_raw = [idx2item[i] for i in range(len(idx2item))]
store.upsert_items("two_tower_items", item_ids_raw, item_embeddings)

print(f"upserted {len(item_ids_raw)} item vectors into 'two_tower_items'")

upserted 3662 item vectors into 'two_tower_items'


## Retrieval + cold-start fallback

Qdrant не умеет сам исключать уже просмотренные товары (в отличие от
`implicit.recommend`), поэтому берём с запасом (`k + число train-позитивов
пользователя`) и фильтруем вручную.

In [4]:
seen_items_by_user = train_positive.groupby("user_id")["item_id"].apply(set)


def two_tower_score_fn(user_id, k):
    user_idx = user2idx[user_id]
    query_vec = user_embeddings[user_idx]
    seen = seen_items_by_user.get(user_id, set())
    hits = store.search("two_tower_items", query_vec, top_k=k + len(seen))
    candidates = [item_id for item_id, _score in hits if item_id not in seen]
    return candidates[:k]


test_users = test_positive["user_id"].unique()
recommended_by_user = {
    user_id: retrieve_candidates(user_id, two_tower_score_fn, popularity_ranking, known_users, k=K)
    for user_id in test_users
}

print(f"построены рекомендации для {len(recommended_by_user)} test-пользователей")

построены рекомендации для 1762 test-пользователей


### Sanity-check cold-start ветки

In [5]:
fake_user_id = -1
assert fake_user_id not in known_users
fallback = retrieve_candidates(fake_user_id, two_tower_score_fn, popularity_ranking, known_users, k=K)
assert fallback == popularity_ranking.head(K).index.tolist()
print("cold-start fallback OK:", fallback[:5])

cold-start fallback OK: [2858, 260, 1196, 2028, 1198]


## Сравнение: Two-Tower vs ALS vs popularity baseline

In [6]:
two_tower_metrics = evaluate_recommendations(recommended_by_user, test_positive, K)
popularity_metrics = evaluate_fixed_recommendation(popularity_ranking.head(K).index.tolist(), test_positive, K)

# ALS-метрики из 02a (пересчитаны здесь же для наглядной сводной таблицы)
from src.id_mapping import build_id_maps
from src.als_model import ALSModel

user_map, item_map = build_id_maps(train)
als = ALSModel(factors=64, regularization=0.01, iterations=15, random_state=0)
als.fit(train_positive, user_map, item_map)


def als_score_fn(user_id, k):
    user_idx = user_map.raw_to_idx[user_id]
    item_idx, _scores = als.recommend_for_user(user_idx, k=k)
    return item_map.to_raw(item_idx)


als_recommended_by_user = {
    user_id: retrieve_candidates(user_id, als_score_fn, popularity_ranking, known_users, k=K)
    for user_id in test_users
}
als_metrics = evaluate_recommendations(als_recommended_by_user, test_positive, K)

summary = pd.DataFrame(
    [popularity_metrics, als_metrics, two_tower_metrics],
    index=["popularity baseline", "ALS + cold-start", "Two-Tower + cold-start"],
)[["recall@k", "ndcg@k", "n_users"]]
summary

  0%|          | 0/15 [00:00<?, ?it/s]

,recall@k,ndcg@k,n_users
popularity baseline,0.243828,0.225475,1762
ALS + cold-start,0.280951,0.256344,1762
Two-Tower + cold-start,0.176769,0.169540,1762
